In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import os

In [2]:
print("Loading data...")
df = pd.read_csv("data/creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

print("data loaded")
print(f"Total: {len(df)}, Fraud: {y.sum()} ({y.mean()*100:.4f}%)")

Loading data...
data loaded
Total: 284807, Fraud: 492 (0.1727%)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Applying SMOTE...")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(
    f"After SMOTE: {len(X_train_res)} samples, fraud ratio: {y_train_res.mean()*100:.2f}%"
)

Applying SMOTE...
After SMOTE: 454902 samples, fraud ratio: 50.00%


In [4]:
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False,
)
model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

c:\Users\ABHI\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:20:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.37      0.87      0.52        98

    accuracy                           1.00     56962
   macro avg       0.69      0.93      0.76     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9769

Confusion Matrix:
[[56722   142]
 [   13    85]]


In [5]:
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/fraud_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(smote, 'models/smote.pkl')

print("\n✅ Models saved to 'models/' folder.")


✅ Models saved to 'models/' folder.


In [6]:
import pandas as pd

df = pd.read_csv('data/creditcard.csv')
fraud_sample = df[df['Class'] == 1].iloc[0]  # pick first fraud case

print("Time:", fraud_sample['Time'])
print("Amount:", fraud_sample['Amount'])
print("V1:", fraud_sample['V1'])
print("V2:", fraud_sample['V2'])
print("V3:", fraud_sample['V3'])
print("V4:", fraud_sample['V4'])
print("V5:", fraud_sample['V5'])
print("V6:", fraud_sample['V6'])
print("V7:", fraud_sample['V7'])
print("V8:", fraud_sample['V8'])
print("V9:", fraud_sample['V9'])
print("V10:", fraud_sample['V10'])

Time: 406.0
Amount: 0.0
V1: -2.3122265423263
V2: 1.95199201064158
V3: -1.60985073229769
V4: 3.9979055875468
V5: -0.522187864667764
V6: -1.42654531920595
V7: -2.53738730624579
V8: 1.39165724829804
V9: -2.77008927719433
V10: -2.77227214465915
